In [1]:
!pip install pennylane

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 49.7 MB/s eta 0:00:00
  Attempting uninstall: autograd
    Found existing installation: autograd 1.9.1
    Uninstalling autograd-1.9.1:
      Successfully uninstalled autograd-1.9.1


In [2]:
import time
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pennylane as qml
import seaborn as sns
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.datasets import load_wine
from sklearn.metrics import (
    accuracy_score,
    adjusted_rand_score,
    confusion_matrix,
    f1_score,
    normalized_mutual_info_score,
    silhouette_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# ==========================================================
# 1. VERİ HAZIRLIĞI (WINE VERİ SETİNE UYARLANDI)
# ==========================================================
print("--- [KOD 6] 9 KÜBİTLİK SWAP-TEST Q-KMEANS (5-FOLD CV) - WINE VERİ SETİ ---")

RANDOM_SEED = 42
wine = load_wine()
X_raw = wine.data
y_true = wine.target

print(f"Toplam Örneklem: {len(X_raw)} (Sınıf 0: {np.sum(y_true == 0)}, Sınıf 1: {np.sum(y_true == 1)}, Sınıf 2: {np.sum(y_true == 2)})")

# ==========================================================
# 2. QPE / SWAP TEST KUANTUM MESAFE DEVRESİ (9 KÜBİT)
# ==========================================================
n_qubits = 4
dev_swap = qml.device("default.qubit", wires=(2 * n_qubits + 1))

@qml.qnode(dev_swap, interface="autograd")
def qpe_swap_distance_circuit(x_vec, c_vec):
    qml.Hadamard(wires=0)
    qml.AngleEmbedding(x_vec, wires=range(1, n_qubits + 1))
    qml.AngleEmbedding(c_vec, wires=range(n_qubits + 1, 2 * n_qubits + 1))
    for i in range(n_qubits):
        qml.CSWAP(wires=[0, i + 1, i + n_qubits + 1])
    qml.Hadamard(wires=0)
    return qml.expval(qml.PauliZ(0))

def compute_quantum_distance(x, c):
    """Fidelity'yi Fubini-Study kuantum mesafesine çevirir: d = sqrt(max(0, 2 - 2*Fidelity))"""
    val = qpe_swap_distance_circuit(x, c)
    fidelity = np.clip(val, 0.0, 1.0)
    return np.sqrt(np.maximum(0.0, 2.0 - 2.0 * fidelity))

# ==========================================================
# 3. ÖZEL KUANTUM Q-KMEANS SINIFI
# ==========================================================
class QuantumKMeans:
    def __init__(self, n_clusters=3, max_iter=5, random_state=42):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.random_state = random_state
        self.centroids = None

    def fit(self, X):
        np.random.seed(self.random_state)
        random_indices = np.random.choice(len(X), self.n_clusters, replace=False)
        self.centroids = X[random_indices].copy()

        labels = np.zeros(len(X))
        for iteration in range(self.max_iter):
            new_labels = np.zeros(len(X))
            for i, x in enumerate(X):
                distances = [compute_quantum_distance(x, c) for c in self.centroids]
                new_labels[i] = np.argmin(distances)

            if np.all(labels == new_labels):
                break
            labels = new_labels

            for k in range(self.n_clusters):
                cluster_points = X[labels == k]
                if len(cluster_points) > 0:
                    self.centroids[k] = cluster_points.mean(axis=0)

        return self

    def predict(self, X):
        preds = np.zeros(len(X))
        for i, x in enumerate(X):
            distances = [compute_quantum_distance(x, c) for c in self.centroids]
            preds[i] = np.argmin(distances)
        return preds.astype(int)

def align_cluster_labels(y_real, y_pred):
    """Wine 3 sınıflı olduğu için Hungarian (Kuhn-Munkres) algoritması ile optimal eşleşme yapar."""
    cm = confusion_matrix(y_real, y_pred)
    cost_matrix = np.max(cm) - cm
    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    mapping = {col: row for row, col in zip(row_ind, col_ind)}
    return np.array([mapping.get(c, c) for c in y_pred])

# ==========================================================
# 4. 5-FOLD STRATIFIED ÇAPRAZ DOĞRULAMA (VERİ SIZINTISIZ)
# ==========================================================
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

metrics_classic = {"acc": [], "f1_bin": [], "f1_macro": [], "sil": [], "ari": [], "nmi": []}
metrics_quantum = {"acc": [], "f1_bin": [], "f1_macro": [], "sil": [], "ari": [], "nmi": []}

start_cv_time = time.time()
print("\n--- 5-Fold Çapraz Doğrulama Başlatılıyor ---")

for fold, (train_idx, val_idx) in enumerate(cv.split(X_raw, y_true), 1):
    f_start = time.time()
    X_tr_raw, y_tr = X_raw[train_idx], y_true[train_idx]
    X_val_raw, y_val = X_raw[val_idx], y_true[val_idx]

    # Data Leakage engelleme: Sadece Train üzerinden fit et!
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr_raw)
    X_val_scaled = scaler.transform(X_val_raw)

    pca = PCA(n_components=n_qubits, random_state=RANDOM_SEED)
    X_tr_4d = pca.fit_transform(X_tr_scaled)
    X_val_4d = pca.transform(X_val_scaled)

    minmax = MinMaxScaler(feature_range=(-np.pi, np.pi))
    X_tr_angles = minmax.fit_transform(X_tr_4d)
    X_val_angles = minmax.transform(X_val_4d)

    # 1. Klasik K-Means
    km_c = KMeans(n_clusters=3, random_state=RANDOM_SEED, n_init=10)
    km_c.fit(X_tr_angles)
    preds_c = align_cluster_labels(y_val, km_c.predict(X_val_angles))

    metrics_classic["acc"].append(accuracy_score(y_val, preds_c))
    metrics_classic["f1_bin"].append(f1_score(y_val, preds_c, average="weighted", zero_division=0))
    metrics_classic["f1_macro"].append(f1_score(y_val, preds_c, average="macro", zero_division=0))
    metrics_classic["sil"].append(silhouette_score(X_val_angles, preds_c))
    metrics_classic["ari"].append(adjusted_rand_score(y_val, preds_c))
    metrics_classic["nmi"].append(normalized_mutual_info_score(y_val, preds_c))

    # 2. Kuantum Swap-Test Q-Means
    qkm = QuantumKMeans(n_clusters=3, max_iter=5, random_state=RANDOM_SEED)
    qkm.fit(X_tr_angles)
    preds_q = align_cluster_labels(y_val, qkm.predict(X_val_angles))

    metrics_quantum["acc"].append(accuracy_score(y_val, preds_q))
    metrics_quantum["f1_bin"].append(f1_score(y_val, preds_q, average="weighted", zero_division=0))
    metrics_quantum["f1_macro"].append(f1_score(y_val, preds_q, average="macro", zero_division=0))
    metrics_quantum["sil"].append(silhouette_score(X_val_angles, preds_q))
    metrics_quantum["ari"].append(adjusted_rand_score(y_val, preds_q))
    metrics_quantum["nmi"].append(normalized_mutual_info_score(y_val, preds_q))

    print(
        f"Fold {fold}/5 Tamamlandı ({time.time() - f_start:.1f} sn) -> "
        f"Klasik F1 (W): %{metrics_classic['f1_bin'][-1]*100:.2f} | Kuantum F1 (W): %{metrics_quantum['f1_bin'][-1]*100:.2f} | ARI: {metrics_quantum['ari'][-1]:.3f}"
    )

total_cv_time = time.time() - start_cv_time

# ==========================================================
# 5. RAPORLAMA TABLOSU
# ==========================================================
print("\n" + "=" * 95)
print("HAKEM RAPORU: 5-FOLD STRATIFIED CV KÜMELEME SONUÇLARI (Mean ± SD)")
print("=" * 95)
print(f"{'Metrik':<28} | {'Klasik K-Means (5-Fold CV)':<30} | {'Kuantum Q-Means (5-Fold CV)'}")
print("-" * 95)
print(f"{'Doğruluk (Acc, %)':<28} | %{np.mean(metrics_classic['acc'])*100:<6.2f} ± {np.std(metrics_classic['acc'])*100:<6.2f}         | %{np.mean(metrics_quantum['acc'])*100:<6.2f} ± {np.std(metrics_quantum['acc'])*100:<6.2f}")
print(f"{'F1-Skoru (Weighted, %)':<28} | %{np.mean(metrics_classic['f1_bin'])*100:<6.2f} ± {np.std(metrics_classic['f1_bin'])*100:<6.2f}         | %{np.mean(metrics_quantum['f1_bin'])*100:<6.2f} ± {np.std(metrics_quantum['f1_bin'])*100:<6.2f}")
print(f"{'F1-Skoru (Macro, %)':<28} | %{np.mean(metrics_classic['f1_macro'])*100:<6.2f} ± {np.std(metrics_classic['f1_macro'])*100:<6.2f}         | %{np.mean(metrics_quantum['f1_macro'])*100:<6.2f} ± {np.std(metrics_quantum['f1_macro'])*100:<6.2f}")
print(f"{'Silüet Skoru':<28} | {np.mean(metrics_classic['sil']):<7.3f} ± {np.std(metrics_classic['sil']):<7.3f}        | {np.mean(metrics_quantum['sil']):<7.3f} ± {np.std(metrics_quantum['sil']):<7.3f}")
print(f"{'ARI Skoru':<28} | {np.mean(metrics_classic['ari']):<7.3f} ± {np.std(metrics_classic['ari']):<7.3f}        | {np.mean(metrics_quantum['ari']):<7.3f} ± {np.std(metrics_quantum['ari']):<7.3f}")
print(f"{'NMI Skoru':<28} | {np.mean(metrics_classic['nmi']):<7.3f} ± {np.std(metrics_classic['nmi']):<7.3f}        | {np.mean(metrics_quantum['nmi']):<7.3f} ± {np.std(metrics_quantum['nmi']):<7.3f}")
print(f"{'Toplam Süre (sn)':<28} | ~0.08 sn                         | {total_cv_time:.1f} sn")
print("=" * 95)

--- [KOD 6] 9 KÜBİTLİK SWAP-TEST Q-KMEANS (5-FOLD CV) - WINE VERİ SETİ ---
Toplam Örneklem: 178 (Sınıf 0: 59, Sınıf 1: 71, Sınıf 2: 48)

--- 5-Fold Çapraz Doğrulama Başlatılıyor ---
Fold 1/5 Tamamlandı (21.9 sn) -> Klasik F1 (W): %94.35 | Kuantum F1 (W): %89.04 | ARI: 0.682
Fold 2/5 Tamamlandı (10.1 sn) -> Klasik F1 (W): %97.23 | Kuantum F1 (W): %91.67 | ARI: 0.760
Fold 3/5 Tamamlandı (10.1 sn) -> Klasik F1 (W): %97.21 | Kuantum F1 (W): %97.21 | ARI: 0.909
Fold 4/5 Tamamlandı (10.1 sn) -> Klasik F1 (W): %97.15 | Kuantum F1 (W): %97.15 | ARI: 0.904
Fold 5/5 Tamamlandı (10.0 sn) -> Klasik F1 (W): %100.00 | Kuantum F1 (W): %94.22 | ARI: 0.820

HAKEM RAPORU: 5-FOLD STRATIFIED CV KÜMELEME SONUÇLARI (Mean ± SD)
Metrik                       | Klasik K-Means (5-Fold CV)     | Kuantum Q-Means (5-Fold CV)
-----------------------------------------------------------------------------------------------
Doğruluk (Acc, %)            | %97.21  ± 1.76           | %93.84  ± 3.22  
F1-Skoru (Weighted, %)